# SmartCampus IA — Test vidéo sur Google Colab
Lance ce notebook sur https://colab.research.google.com
**Runtime → Change runtime type → CPU** (suffisant pour InsightFace)

In [ ]:
# 1. Installer les dépendances
!pip install -q insightface onnxruntime opencv-python-headless psycopg2-binary sqlalchemy pgvector numpy

In [ ]:
# 2. Monter Google Drive (pour accéder à la vidéo et au code)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Copier le dossier backend depuis Drive
# Change le chemin si nécessaire
import shutil, os

BACKEND_DRIVE = '/content/drive/MyDrive/ProjetPFA/backend'  # <-- adapte ce chemin
BACKEND_LOCAL = '/content/backend'

if os.path.exists(BACKEND_LOCAL):
    shutil.rmtree(BACKEND_LOCAL)
shutil.copytree(BACKEND_DRIVE, BACKEND_LOCAL)
print(f'Backend copié : {os.listdir(BACKEND_LOCAL)}')

In [ ]:
# 4. Chemin de la vidéo (sur Drive ou uploadée directement)
VIDEO_PATH = '/content/drive/MyDrive/ProjetPFA/video_classe.mp4'  # <-- adapte

# OU upload direct depuis ton PC :
# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = list(uploaded.keys())[0]

import os
print(f'Vidéo : {VIDEO_PATH}  —  existe : {os.path.isfile(VIDEO_PATH)}')

In [ ]:
# 5. Lancer la reconnaissance (mode headless + save)
import subprocess, sys

cmd = [
    sys.executable,
    f'{BACKEND_LOCAL}/video_attendance.py',
    VIDEO_PATH,
    '--offline',    # utilise embeddings_cache.json (pas besoin de connexion DB)
    '--headless',   # pas d'affichage GUI
    '--save',       # sauvegarde les captures annotées
    '--interval', '3',  # analyse 1 frame toutes les 3 secondes
]

result = subprocess.run(cmd, capture_output=False, text=True, cwd=BACKEND_LOCAL)
print('\nCode retour :', result.returncode)

In [ ]:
# 6. Afficher les captures annotées
import glob
from IPython.display import Image, display

captures_dir = os.path.join(os.path.dirname(VIDEO_PATH), 'captures_annotees')
images = sorted(glob.glob(f'{captures_dir}/*.jpg'))
print(f'{len(images)} captures sauvegardées')
for img_path in images[:5]:  # affiche les 5 premières
    print(img_path)
    display(Image(img_path, width=800))

In [ ]:
# 7. Télécharger toutes les captures sur ton PC
import shutil
from google.colab import files

zip_path = '/content/captures_annotees.zip'
shutil.make_archive('/content/captures_annotees', 'zip', captures_dir)
files.download(zip_path)